# Session 1: Ohm's law and electrical power
**A. Real-world problem.** A phone charger, a torch bulb and a toaster all push electric
current through something that resists it. How much current flows, and how much power turns
into heat? Two short rules answer both.

**Plan.** (1) Ohm's law V = I R, (2) power P = V I, (3) three graphs: I vs V, I vs R, P vs R.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: Ohm's law
Voltage V (volts) is the push, resistance R (ohms) is how hard the part resists, and the
current I (amperes) is the flow. For an *ohmic* part, I = V / R.

Check by hand: V = 12 V across R = 6 ohm gives I = 12 / 6 = 2 A.

In [ ]:
def current(volts, ohms):
    """Current in amperes through a resistor (ideal supply, ideal wires)."""
    if ohms <= 0:
        raise ValueError("resistance must be positive")
    return volts / ohms

## Step 2: electrical power
Power is energy per second turned into heat in the resistor: P = V I. Using Ohm's law it
can also be written P = I^2 R and P = V^2 / R. All three give the same number.

Check by hand: 12 V and 2 A give P = 24 W. Also I^2 R = 4 * 6 = 24 W.

In [ ]:
def power(volts, ohms):
    """Power in watts dissipated in the resistor, P = V^2 / R."""
    return volts * volts / ohms


def power_forms(volts, ohms):
    """The three equivalent forms (V*I, I^2*R, V^2/R), in watts."""
    i = current(volts, ohms)
    return volts * i, i * i * ohms, volts * volts / ohms

## Step 3: see it
Three graphs, one question each. I vs V: a straight line with slope 1/R. I vs R: a falling
curve (more resistance, less current). P vs R: also falling at fixed V.

In [ ]:
PRESET_V = (1.0, 12.0, 24.0)
PRESET_R = (1.0, 6.0, 12.0, 24.0)


def table(volts_list=PRESET_V, ohms_list=PRESET_R):
    """Rows (V, R, I, P) for every preset combination."""
    return [(v, r, current(v, r), power(v, r)) for v in volts_list for r in ohms_list]


if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    for v, r, i, p in table((12.0,), PRESET_R):
        print(f"V = {v:4.0f} V, R = {r:4.0f} ohm -> I = {i:.3f} A, P = {p:.1f} W")
    print("24 ohm on 12 V draws", current(12.0, 24.0), "A")

    v = np.linspace(0, 24, 100)
    fig, ax = plt.subplots(figsize=(6, 4))
    for r in PRESET_R:
        ax.plot(v, v / r, label=f"R = {r:g} ohm")
    ax.set_xlabel("voltage V (volts)")
    ax.set_ylabel("current I (amperes)")
    ax.set_title("I versus V: straight lines, slope = 1/R")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "i_vs_v.png", dpi=120)

    r = np.linspace(1, 24, 100)
    fig2, (a1, a2) = plt.subplots(1, 2, figsize=(9, 4))
    for volts in PRESET_V:
        a1.plot(r, volts / r, label=f"V = {volts:g} V")
        a2.plot(r, volts ** 2 / r, label=f"V = {volts:g} V")
    a1.set_xlabel("resistance R (ohms)")
    a1.set_ylabel("current I (amperes)")
    a1.set_title("I versus R")
    a2.set_xlabel("resistance R (ohms)")
    a2.set_ylabel("power P (watts)")
    a2.set_title("P versus R")
    for a in (a1, a2):
        a.grid(alpha=0.3)
        a.legend()
    a2.set_ylim(0, 150)
    fig2.tight_layout()
    fig2.savefig(OUT / "i_p_vs_r.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Run `table()` in `lesson.py` and predict, before looking, which of the 12 preset combinations has the largest power.
2. **Stretch.** Write `voltage_for_power(p_watts, ohms)` that returns the voltage which makes a resistor dissipate a given power, and test it with 24 W and 6 ohm (expect 12 V).
3. **Explain.** Explain why a short circuit (R close to zero) on a battery is dangerous, using P = V^2 / R.

## Assessment
- **A. Conceptual.** If you double the voltage across a resistor, what happens to the current and to the power? Explain why they change by different factors.
- **B. Numerical.** What current does a 24 ohm resistor draw from an ideal 12 V supply, and how much power does it dissipate? Do a sanity check.
- **C. Graph and simulation interpretation.** On the P versus R graph at 12 V, what is P at 6, 12 and 24 ohm? What does the pattern tell you about halving the resistance?
- **D. Design and optimisation challenge.** Design: from the preset resistors 1, 6, 12 and 24 ohm, choose one for a 12 V supply so that the current is at most 1 A and the resistor dissipates at most 10 W. What is the smallest resistance, not limited to presets, that meets both limits?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: 12 V across 6 ohm gives 2 A and 24 W",
                   abs(L.current(12, 6) - 2.0) < 1e-12 and abs(L.power(12, 6) - 24.0) < 1e-12))
    checks.append(("hand calculation: 24 ohm on 12 V draws 0.5 A", abs(L.current(12, 24) - 0.5) < 1e-12))
    a, b, c = L.power_forms(7.5, 3.0)
    checks.append(("V*I, I^2*R and V^2/R agree", np.allclose([a, b], c)))
    checks.append(("doubling V doubles I and quadruples P",
                   abs(L.current(2, 5) / L.current(1, 5) - 2) < 1e-12 and abs(L.power(2, 5) / L.power(1, 5) - 4) < 1e-12))
    checks.append(("doubling R halves I", abs(L.current(12, 12) / L.current(12, 6) - 0.5) < 1e-12))
    try:
        L.current(12, 0)
        ok = False
    except ValueError:
        ok = True
    checks.append(("zero resistance is rejected (short circuit, not a number)", ok))
    checks.append(("table has 12 preset rows", len(L.table()) == 12))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
